In [ ]:
# Cell 1: API Call and Environment Setup
# This cell sets up the environment by loading various API keys securely from Google Colab's secrets and installing necessary packages.
import os
from google.colab import userdata

# Securely load the API keys using try...except blocks to gracefully handle any missing keys
# We check each key individually so that the app can still run using fallback models if one key is missing.
try:
    os.environ["OPENCODE_API_KEY"] = userdata.get('OPENCODE_API_KEY')
    print("OPENCODE_API_KEY loaded.")
except Exception as e:
    print(f"Warning: Could not load OPENCODE_API_KEY. ({e})")

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("GROQ_API_KEY loaded.")
except Exception as e:
    print(f"Warning: Could not load GROQ_API_KEY. ({e})")

try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("GEMINI_API_KEY loaded.")
except Exception as e:
    print(f"Warning: Could not load GEMINI_API_KEY. ({e})")

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("GOOGLE_API_KEY loaded.")
except Exception as e:
    print(f"Warning: Could not load GOOGLE_API_KEY. ({e})")

# Updated dependencies to include openai, google-genai, and groq packages for LLM integration
!pip install -q langchain langchain-core langchain-openai langchain-google-genai langchain-groq langgraph gradio

OPENCODE_API_KEY loaded.
GROQ_API_KEY loaded.
GOOGLE_API_KEY loaded.


In [ ]:
# Cell 2: All Imports, Data Loading & LangChain Tools
# This cell mounts Google Drive, loads the JSON databases (hotels, locations, events), and defines the specialized LangChain tools the AI agent will use.
import os
import json
import re
import math
import warnings
import gradio as gr
from typing import List
from google.colab import userdata
from google.colab import drive
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

warnings.filterwarnings("ignore")

# Mount Google Drive to access the JSON data files stored in the user's workspace
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define the directory where data files are located
folder_path = '/content/drive/MyDrive/ParisTravelPlanner/'

class ParisDataLoader:
    # A helper class to load the local JSON datasets into memory for fast tool access
    def __init__(self):
        try:
            # Read all JSON files into dictionaries/lists for quick lookup
            with open(os.path.join(folder_path, 'hotels.json'), 'r') as f:
                self.hotels = json.load(f)
            with open(os.path.join(folder_path, 'locations.json'), 'r') as f:
                self.locations = json.load(f)
            with open(os.path.join(folder_path, 'events.json'), 'r') as f:
                self.events = json.load(f)
            print(f"Successfully loaded {len(self.hotels)} hotels, "
                  f"{len(self.locations)} locations, "
                  f"{len(self.events)} events from {folder_path}")
        except FileNotFoundError as e:
            print(f"Error: Could not find JSON files in {folder_path}.")
            print("Ensure the folder 'ParisTravelPlanner' exists in MyDrive with the 3 JSON files.")
            raise e

# Instantiate the dataloader so tools can use the data globally
data_loader = ParisDataLoader()

# ── Tool 1: Hotel Search ─────────────────────────────────────────────────────
@tool
def search_hotels(budget: int) -> str:
    """
    Searches for hotels within the nightly budget in euros.
    Pass only the budget as an integer, e.g. 200
    Returns the top 5 hotels sorted by rating with name, stars, price, style, and amenities.
    """
    # Filter hotels by price limit and ensure a minimum quality rating of 4.0
    results = [
        h for h in data_loader.hotels
        if h.get('price_per_night', float('inf')) <= budget and h.get('rating', 0) >= 4.0
    ]
    # Sort the valid hotels highest-rating first, then cap at 5 results
    results = sorted(results, key=lambda x: x.get('rating', 0), reverse=True)[:5]

    if not results:
        return f"No hotels found under {budget} euros per night. Try a higher budget."

    # Format the dictionary items into a JSON string for the agent
    return json.dumps([{
        "name": h.get('name', 'Unknown'),
        "stars": h.get('stars', 'N/A'),
        "price_per_night": h.get('price_per_night', 'N/A'),
        "rating": h.get('rating', 'N/A'),
        "style": h.get('style', 'N/A'),
        "amenities": h.get('amenities', []),
        "arrondissement": h.get('arrondissement', 'N/A')
    } for h in results])

# ── Tool 2: Location Details ───────────────────────────────────────────────
@tool
def get_location_details(query: str) -> str:
    """
    Checks opening hours, closed_on days, entry fees, and average visit duration for a location.
    ALWAYS call this before scheduling any location to catch closure conflicts.
    Pass the location name or a category like 'museum', 'park', or 'landmark'.
    """
    # Match user query against the location's actual name or its list of descriptive tags
    results = [
        loc for loc in data_loader.locations
        if query.lower() in loc.get('name', '').lower()
        or query.lower() in [t.lower() for t in loc.get('tags', [])]
    ]

    if not results:
        return f"No location found for query: '{query}'"

    # Return up to 3 matches with structural data to help the agent plan time/money
    matches = []
    for loc in results[:3]:
        matches.append({
            "name": loc.get('name', 'Unknown'),
            "type": loc.get('type', 'Unknown'),
            "entry_fee": loc.get('entry_fee', 0),
            "avg_duration_hours": loc.get('avg_duration_hours', 1.0),
            "opening_hours": loc.get('opening_hours', 'Unknown'),
            "closed_on": loc.get('closed_on', []),
            "rating": loc.get('rating', 'N/A')
        })
    return json.dumps(matches)

# ── Tool 3: Search by Interest Tags ───────────────────────────────────────
@tool
def search_locations_by_tags(tags: List[str]) -> str:
    """
    Find locations that match the user's interests and preferences.
    Pass a list of interest tags (e.g. ['art', 'history', 'outdoor']).
    Results are sorted by tag match count, then by rating.
    """
    tag_list = [t.strip().lower() for t in tags]

    # Score each location based on how many tags it overlaps with
    scored = []
    for loc in data_loader.locations:
        loc_tags = [t.lower() for t in loc.get('tags', [])]
        loc_type = loc.get('type', '').lower()
        match_count = sum(1 for t in tag_list if t in loc_tags or t in loc_type)
        if match_count > 0:
            scored.append((match_count, loc.get('rating', 0), loc))

    # Sort by the number of matching tags, then by overall location rating
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    top = [item[2] for item in scored[:8]]

    if not top:
        return f"No locations found matching tags: {tags}. Try broader terms like 'art', 'history', or 'outdoor'."

    return json.dumps([{
        "name": loc.get('name', 'Unknown'),
        "type": loc.get('type', 'Unknown'),
        "tags": loc.get('tags', []),
        "entry_fee": loc.get('entry_fee', 0),
        "avg_duration_hours": loc.get('avg_duration_hours', 1.0),
        "closed_on": loc.get('closed_on', []),
        "rating": loc.get('rating', 'N/A')
    } for loc in top])

# ── Tool 4: Events Search ─────────────────────────────────────────────────────
@tool
def search_events(travel_month: str) -> str:
    """
    Retrieves upcoming events in Paris filtered by travel month.
    Pass the month name, e.g. May
    """
    MONTHS = {
        "january": "01", "february": "02", "march": "03", "april": "04",
        "may": "05", "june": "06", "july": "07", "august": "08",
        "september": "09", "october": "10", "november": "11", "december": "12"
    }

    events = data_loader.events
    # Map the text string of the month to a numerical format often found in dates
    month_num = MONTHS.get(travel_month.strip().lower())
    if month_num:
        # Check if '-MM-' appears in the event date string
        events = [e for e in events if f"-{month_num}-" in e.get('date', '')]

    if not events:
        return f"No events found for: '{travel_month}'."

    return json.dumps([{
        "title": e.get('title', 'Unknown Title'),
        "date": e.get('date', 'Unknown Date'),
        "price": e.get('price', 0),
        "description": e.get('description', '')
    } for e in events[:15]])

# ── Tool 5: Daily Cost Estimator ────────────────────────────────────────────
@tool
def estimate_day_cost(location_names: List[str]) -> str:
    """
    Given a list of location names planned for a single day,
    returns the total entry fee cost and estimated total hours needed.
    Use this to verify a day's plan is realistic and fits the user's activity budget.
    Note: a realistic day should not exceed 8-9 hours of activities.
    """
    names = [n.strip() for n in location_names]
    total_fee = 0
    total_hours = 0.0
    found = []
    not_found = []

    for name in names:
        # Look for the exact matching name in our loaded dataset
        match = next(
            (loc for loc in data_loader.locations if name.lower() in loc.get('name', '').lower()),
            None
        )
        if match:
            # Accumulate the fees and required visit durations
            total_fee += match.get('entry_fee', 0)
            total_hours += match.get('avg_duration_hours', 1.0)
            found.append({
                "name": match.get('name', 'Unknown'),
                "entry_fee": match.get('entry_fee', 0),
                "duration_hours": match.get('avg_duration_hours', 1.0)
            })
        else:
            not_found.append(name)

    # Generate a warning dynamically if the requested itinerary is physically exhausting
    warning = " WARNING: Over 9 hours — consider removing one location." if total_hours > 9 else ""

    result = {
        "locations": found,
        "total_entry_fees_euros": total_fee,
        "estimated_total_hours": round(total_hours, 1),
        "feasibility_note": f"{'Realistic schedule.' if total_hours <= 9 else 'Too many locations.'}{warning}",
        "not_found": not_found
    }
    return json.dumps(result)

# Aggregate all our registered functions into a list for the LangGraph agent
tools = [search_hotels, get_location_details, search_locations_by_tags, search_events, estimate_day_cost]
print(f"\nAll tools initialized: {[t.name for t in tools]}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully loaded 30 hotels, 30 locations, 90 events from /content/drive/MyDrive/ParisTravelPlanner/

All tools initialized: ['search_hotels', 'get_location_details', 'search_locations_by_tags', 'search_events', 'estimate_day_cost']


In [ ]:
# Cell 3: Route Optimizer (Nearest-Neighbor Greedy + 2-opt Improvement)
# This cell defines an optimization algorithm to order the daily itinerary locations efficiently and minimize total walking distance.

def calculate_distance(coord1, coord2):
    """
    Haversine-approximation distance between two GPS coordinates.
    Scales longitude by cos(48.85 degrees) = 0.66 to account for Paris latitude.
    Returns a value proportional to kilometers. This is a fast, simplified distance measure.
    """
    dlat = coord1.get('lat', 0) - coord2.get('lat', 0)
    dlng = (coord1.get('lng', 0) - coord2.get('lng', 0)) * 0.66  # latitude correction for Paris
    return math.sqrt(dlat**2 + dlng**2)

def total_route_distance(route):
    """Sum of distances along an ordered list of locations. Iterates consecutively through the route array."""
    return sum(
        calculate_distance(route[i].get('coords', {}), route[i+1].get('coords', {}))
        for i in range(len(route) - 1)
    )

@tool
def optimize_route(location_names: list[str]) -> str:
    """
    Pass a list of EXACT location names to find the optimal visit order for a single day.
    Uses a nearest-neighbor greedy algorithm seeded at the highest-rated location,
    then applies a 2-opt improvement pass to further reduce total walking distance.
    Run this once per day with that day's planned locations.
    """
    names = [n.strip() for n in location_names]
    # Fetch the underlying location dictionary items from our dataset so we have coordinates
    candidate_locs = [loc for loc in data_loader.locations if loc.get('name') in names]

    if not candidate_locs:
        return "Could not find the specified locations. Make sure names exactly match the dataset."

    if len(candidate_locs) == 1:
        return f"Optimized Route: {candidate_locs[0].get('name', 'Unknown')}"

    # Step 1: Greedy nearest-neighbor — start at the "highest-value" location
    # We define "highest value" as rating multiplied by popularity to act as an anchor.
    remaining = list(candidate_locs)
    current = max(remaining, key=lambda x: x.get('rating', 0) * x.get('popularity_score', 0))
    route = [current]
    remaining.remove(current)

    # Iteratively pick the closest location to the current one, factoring in a small penalty/bonus for popularity
    while remaining:
        next_loc = min(
            remaining,
            key=lambda x: calculate_distance(current.get('coords', {}), x.get('coords', {})) - (x.get('popularity_score', 0) * 0.05)
        )
        route.append(next_loc)
        remaining.remove(next_loc)
        current = next_loc

    # Step 2: 2-opt improvement — try reversing segments to reduce total distance to prevent crossing paths
    # This iterates over pairs of points and swaps edges if it reduces total cost (distance).
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 1):
            for j in range(i + 1, len(route)):
                candidate = route[:i] + route[i:j+1][::-1] + route[j+1:]
                if total_route_distance(candidate) < total_route_distance(route):
                    route = candidate
                    improved = True

    # Convert the sorted location dictionaries back into a simple string of names
    route_names = [loc.get('name', 'Unknown') for loc in route]
    # Approximate the Haversine distance unit back into rough kilometers (1 degree lat ~= 111km)
    approx_km = round(total_route_distance(route) * 111, 1)
    return f"Optimized Route: {' → '.join(route_names)} (approx. {approx_km} km total walking)"

# Add the optimizer to our main tool list
tools.append(optimize_route)
print(f"Route optimizer loaded. Total tools available: {len(tools)}")
print(f"Tools: {[t.name for t in tools]}")

Route optimizer loaded. Total tools available: 6
Tools: ['search_hotels', 'get_location_details', 'search_locations_by_tags', 'search_events', 'estimate_day_cost', 'optimize_route']


In [ ]:
# Cell 4: LLM Reasoning Core & Agent Assembly with Memory and Fallbacks
# This cell configures a multi-model fallback chain checking OpenCode first, sets the system prompt, and initializes the LangGraph agent with conversational memory.
import os

# Initialize an empty list that will hold our ordered preference of Large Language Models
llm_chain = []

# 1. Primary Models: OpenCode (Requires OPENCODE_API_KEY and specific base_url)
# Try to initialize OpenCode as our top-priority choice if the key exists.
opencode_key = os.environ.get("OPENCODE_API_KEY")
if opencode_key and opencode_key != "missing_key":
    llm_chain.append(ChatOpenAI(
        model="gpt-5.5",
        api_key=opencode_key,
        base_url="https://opencode.ai/zen/v1",
        temperature=0.2,
        max_tokens=4000,
        max_retries=1
    ))
    llm_chain.append(ChatOpenAI(
        model="claude-opus-4-7",
        api_key=opencode_key,
        base_url="https://opencode.ai/zen/v1",
        temperature=0.2,
        max_tokens=4000,
        max_retries=1
    ))
    print("Added OpenCode models to the chain.")

# 2. Fallback: Groq
# If OpenCode models fail (e.g. rate limits), Groq's fast Llama instance acts as a backup.
groq_key = os.environ.get("GROQ_API_KEY")
if groq_key and groq_key != "missing_key":
    llm_chain.append(ChatGroq(
        model="llama-3.3-70b-versatile",
        api_key=groq_key,
        temperature=0.2,
        max_tokens=4000,
        max_retries=1
    ))
    print("Added Groq model to the chain.")

# 3. Fallbacks: Gemini & Google
# Further backups if needed
gemini_key = os.environ.get("GEMINI_API_KEY")
if gemini_key and gemini_key != "missing_key":
    llm_chain.append(ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        api_key=gemini_key,
        temperature=0.2,
        max_tokens=4000,
        max_retries=1
    ))
    print("Added Gemini model to the chain.")

google_key = os.environ.get("GOOGLE_API_KEY")
if google_key and google_key != "missing_key":
    llm_chain.append(ChatGoogleGenerativeAI(
        model="gemini-1.5-pro",
        api_key=google_key,
        temperature=0.2,
        max_tokens=4000,
        max_retries=1
    ))
    print("Added Google model to the chain.")

# Validate and wrap models into a fallback chain that LangChain understands
if not llm_chain:
    print("Warning: No valid API keys found. Agent may fail to invoke.")
    # Provide a dummy fallback to prevent initialization crashes when completely empty
    llm_with_fallbacks = ChatOpenAI(api_key="dummy", max_tokens=4000)
else:
    # llm_chain[0] is the primary model, the rest are chained as backups
    llm_primary = llm_chain[0]
    llm_with_fallbacks = llm_primary.with_fallbacks(llm_chain[1:]) if len(llm_chain) > 1 else llm_primary

# Master System Prompt outlining rules the agent MUST follow for itinerary creation
system_prompt = """You are an expert Paris Travel Planner. Your goal is to create a complete, personalized 5-day itinerary based on the user's preferences, budget, and constraints.

STRICT PLANNING RULES — follow these in order:
1. Call 'search_events' with the travel month first to find any special events.
2. Call 'search_locations_by_tags' using the user's stated interests.
3. Call 'get_location_details' on any location before scheduling it — check 'closed_on' and hours. Never schedule a location on its 'closed_on' day.
4. Call 'search_hotels' to find accommodations within budget. If no hotels are found under budget, suggest the closest available option and inform the user.
5. Geographic Clustering: Group locations that are in the same or adjacent areas on the same day to minimize transit time.
6. Plan each day with a MAXIMUM of 3-4 locations (a full day is ~8 hours). Allocate specific times for lunch and dinner breaks.
7. Pacing constraints: Spread high-duration or physically demanding activities across different days to avoid fatigue.
8. Call 'estimate_day_cost' for each day to confirm the schedule is realistic.
9. Call 'optimize_route' for each day's locations to get the best walking order.
10. Respect all mobility constraints when selecting locations.
11. Include entry fees, opening times, and route order in the final itinerary.
12. At the end of every response, ALWAYS provide 3 suggested follow-up questions or actions the user can take, formatted as a markdown bulleted list under the heading '### Suggested Follow-ups'.

Output a beautifully formatted day-by-day itinerary including: recommended hotels at the top, each day's schedule with times and entry fees, daily route order, any special events, a total estimated cost summary, and the suggested follow-ups."""

# Initialize the memory checkpointer to store conversational context for LangGraph
memory = MemorySaver()

# Using LangGraph's prebuilt React agent with memory and our fallback LLM configuration
# This binds the prompt, tools, LLM, and conversational memory states together.
agent_executor = create_react_agent(
    llm_with_fallbacks,
    tools,
    prompt=system_prompt,
    checkpointer=memory
)

def chat_with_agent(user_message: str, history: list) -> str:
    """Handles conversational turns using LangGraph's checkpointer thread."""
    # Thread ID allows LangGraph to fetch earlier messages from the same chat session
    config = {"configurable": {"thread_id": "session_2"}}
    try:
        # Invoke the agent, letting it autonomously decide which tools to use to answer the message
        response = agent_executor.invoke(
            {"messages": [("user", user_message)]},
            config=config
        )
        # Return the final message content generated by the agent
        return response["messages"][-1].content
    except Exception as e:
        return f"An error occurred during planning: {str(e)}\n\nPlease verify your API keys and try again."

Added OpenCode models to the chain.
Added Groq model to the chain.
Added Google model to the chain.


In [ ]:
# Troubleshooting OpenCode API Connection
# This is a diagnostic script to directly verify our LLM API key logic is working independently of the complex agent overhead.
import os
from langchain_openai import ChatOpenAI

print("Testing OpenCode API Connection...")
test_key = os.environ.get("OPENCODE_API_KEY")

if not test_key or test_key == "missing_key":
    print("Error: OPENCODE_API_KEY is not set or is missing in the environment.")
else:
    try:
        # Initialize the test LLM with the exact parameters used in the main chain
        # This isolates just the model connection parameters.
        test_llm = ChatOpenAI(
            model="gpt-5.5",
            api_key=test_key,
            base_url="https://opencode.ai/zen/v1",
            max_retries=1,
            timeout=15.0 # Set a timeout so it doesn't hang indefinitely
        )

        print(f"Attempting to contact https://opencode.ai/zen/v1 with model gpt-5.5...")
        # Invoke a simple prompt to force a network request
        response = test_llm.invoke("Hello, are you receiving this? Reply with 'Connection successful.'")

        print("\n✅ SUCCESS! Response from OpenCode API:")
        print(response.content)

    except Exception as e:
        print("\n❌ FAILED to connect to OpenCode API. Error details:")
        print(e)

Testing OpenCode API Connection...
Attempting to contact https://opencode.ai/zen/v1 with model gpt-5.5...

✅ SUCCESS! Response from OpenCode API:
Connection successful.


In [ ]:
# Cell 5: Gradio Front-End with Chat Interface
# This cell creates a public Gradio UI to interact with the travel planner agent in a conversational format.
import gradio as gr

# Gr.Blocks is the low-level UI container that lets us mix Markdown text and interactive elements.
with gr.Blocks() as demo:
    gr.Markdown("# Paris Travel Planner AI Agent (Conversational)")
    gr.Markdown(
        "Chat with the agent to build and refine your itinerary. "
        "You can start by providing your dates, budget, mobility needs, and preferences. "
        "Then, ask follow-up questions to adjust the plan!"
    )

    # Using Gradio's Chatbot component, linked to our LangGraph handler function 'chat_with_agent'
    # This handles the text input and automatically displays the back-and-forth chat history.
    chatbot = gr.ChatInterface(
        fn=chat_with_agent,
        examples=[
            "Plan a 5-day trip for me this May. My budget is 250 euros per night. I love art and wine!",
            "Can you change the second day to include more outdoor parks?",
            "What are some cheaper hotel alternatives?"
        ]
    )

# share=True is required for Colab to expose the Gradio interface publicly via an external link
# debug=True ensures backend error traces are visible here in the cell output.
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://afdf96b2af14aab21b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://afdf96b2af14aab21b.gradio.live
